In [1]:
import time, json, re, numpy as np, pandas as pd, torch
from sentence_transformers import SentenceTransformer
from scipy.stats import spearmanr
from m1_utils import *

EMB, TAG, PV = "BAAI/bge-small-en-v1.5", "bge", "v1"
QP = "Represent this sentence for searching relevant passages: "

queries, judg, corpus = load_all(); qrels = build_qrels(judg); pids = corpus["product_id"].to_numpy()
eval_q = pd.read_parquet(OUT / "eval_query_order.parquet")

def load_cache(path):
    d = {}
    for line in path.open(encoding="utf-8"):
        r = json.loads(line); d[r["query_id"]] = r["text"]
    return d

def boot(a, b, nb=5000, seed=0):
    d = np.asarray(a) - np.asarray(b); r = np.random.default_rng(seed)
    m = np.array([d[r.integers(0, len(d), len(d))].mean() for _ in range(nb)])
    return d.mean(), np.percentile(m, [2.5, 97.5])

device = "cuda" if torch.cuda.is_available() else "cpu"; print("device:", device)
st = SentenceTransformer(EMB, device=device); st.max_seq_length = 256
t0 = time.time()
doc_emb = st.encode(corpus["doc_text"].tolist(), batch_size=128, normalize_embeddings=True,
                    show_progress_bar=True, convert_to_numpy=True).astype("float32")
np.save(OUT / f"doc_emb_{TAG}.npy", doc_emb)
print(doc_emb.shape, f"{time.time()-t0:.0f}s")

for t in ["lg tone triumph", "wireless earbuds for running", "ssd rgb"]:
    q = st.encode([QP + t], normalize_embeddings=True, convert_to_numpy=True)[0]
    top = np.argsort(-(doc_emb @ q))[:3]
    print("\n", t); [print("   ", corpus["product_title"].iloc[i][:90]) for i in top]

device: cpu


Loading weights:   0%|          | 0/199 [00:00<?, ?it/s]

Batches:   0%|          | 0/401 [00:00<?, ?it/s]

(51318, 384) 4821s

 lg tone triumph
    LG TONE TRIUMPH HBS-510 wireless Bluetooth headset - Black
    LG Tone HBS-510 Triumph Black - Bluetooth Wireless Stereo Headset 510 with 1.2Amp Quick Wa
    LG HBS-510.ACUSBKI Tone Triumph HBS-510 Wireless Bluetooth Headset Black Bundle with Deco 

 wireless earbuds for running
    Wireless Earbuds Bluetooth Headphones Noise Cancelling Wireless Earphones Sport Earbuds wi
    Treblab XR700 - Wireless Running Earbuds - Top Sports Headphones, Custom Adjustable Earhoo
    Wireless Headphones | Best Wireless Earbuds |Wireless Workout Earphones | Running Workout 

 ssd rgb
    Savant PCs A o r u s RGB Logo 2.5" Hard Drive (HDD) or Solid State Drive (SSD) Cover
    HyperX Fury RGB SSD 240GB SATA 3 2.5" Solid State Drive Black Case with Multi-Color RGB SH
    Savant PCs R O G Logo RGB 2.5" Hard Drive (HDD) or Solid State Drive (SSD) Cover


In [3]:
bm25_all = np.load(OUT / "bm25_scores.npy", mmap_mode="r")
P = pd.read_parquet(OUT / "clf_probs_eval.parquet"); assert (P["query_id"].values == eval_q["query_id"].values).all()
alphas = np.round(np.linspace(0, 1, 11), 1); K = 100

def build_vecs(df, role):
    qids = df["query_id"].tolist()
    prose = load_cache(OUT / f"hyde_{role}_prose_{PV}.jsonl"); spec = load_cache(OUT / f"hyde_{role}_spec_{PV}.jsonl")
    assert all(q in prose and q in spec for q in qids)
    enc = lambda tx: st.encode(tx, batch_size=64, normalize_embeddings=True, convert_to_numpy=True).astype("float32")
    qr = enc([QP + t for t in df["query"]]); qp = enc([prose[q] for q in qids]); qs = enc([spec[q] for q in qids])
    return {"raw": qr, "prose": qp, "spec": qs, "prose_q": (qr + qp) / 2, "spec_q": (qr + qs) / 2}

def top10(s):
    t = np.argpartition(-s, 10)[:10]; return list(pids[t[np.argsort(-s[t])]])

def run(rows, qids, VEC, vec=None, alpha=None):
    ranked = {}
    for j, q in enumerate(qids):
        b = np.asarray(bm25_all[rows[j]])
        if vec is None: ranked[q] = top10(b); continue
        d = VEC[vec][j] @ doc_emb.T
        if alpha is None: ranked[q] = top10(d); continue
        cand = np.union1d(np.argpartition(-b, K)[:K], np.argpartition(-d, K)[:K])
        s = alpha[j] * minmax(b[cand]) + (1 - alpha[j]) * minmax(d[cand])
        ranked[q] = list(pids[cand[np.argsort(-s)[:10]]])
    return evaluate(ranked, qrels)

dev = eval_q[eval_q["role"] == "dev"]; drows = dev.index.to_numpy(); dq = dev["query_id"].tolist(); nd = len(dq)
VD = build_vecs(dev, "dev")

print("DENSE-ONLY, dev NDCG@10 (bge):")
print(pd.Series({v: run(drows, dq, VD, v)["ndcg@10"].mean() for v in VD}).round(4).to_string())

curves = {v: [run(drows, dq, VD, v, np.full(nd, a))["ndcg@10"].mean() for a in alphas] for v in ["raw", "prose_q", "spec_q"]}
cv = pd.DataFrame(curves, index=alphas); print("\nFUSED with BM25, dev NDCG@10 by fixed alpha:"); print(cv.round(4).to_string())
A_BGE = float(alphas[int(np.argmax(cv.mean(axis=1).values))]); print("\nA_BGE (frozen for test) =", A_BGE)

# alpha diagnostic with the classifier (predicted P(Exact), temperature-scaled)
pE = P.loc[drows, "pE_T"].values
fixed = run(drows, dq, VD, "spec_q", np.full(nd, A_BGE))["ndcg@10"].values
formula = run(drows, dq, VD, "spec_q", pE)["ndcg@10"].values
adv = (run(drows, dq, VD, "spec_q", np.full(nd, .9))["ndcg@10"].values - run(drows, dq, VD, "spec_q", np.full(nd, .1))["ndcg@10"].values)
rho, p = spearmanr(pE, adv)
m, ci = boot(formula, fixed)
print(f"\nalpha=P(Exact) {formula.mean():.4f} vs fixed {fixed.mean():.4f}: diff {m:+.4f} CI [{ci[0]:+.4f},{ci[1]:+.4f}]")
print(f"rho(P_exact, BM25 advantage) = {rho:.3f} (p={p:.3f})")

DENSE-ONLY, dev NDCG@10 (bge):
raw        0.2973
prose      0.2570
spec       0.2375
prose_q    0.3036
spec_q     0.3076

FUSED with BM25, dev NDCG@10 by fixed alpha:
        raw  prose_q  spec_q
0.0  0.2973   0.3036  0.3076
0.1  0.3089   0.3185  0.3290
0.2  0.3149   0.3242  0.3383
0.3  0.3204   0.3321  0.3407
0.4  0.3166   0.3263  0.3381
0.5  0.3117   0.3203  0.3286
0.6  0.3014   0.3081  0.3153
0.7  0.2887   0.2930  0.3004
0.8  0.2756   0.2787  0.2804
0.9  0.2630   0.2659  0.2654
1.0  0.2501   0.2503  0.2500

A_BGE (frozen for test) = 0.3

alpha=P(Exact) 0.3169 vs fixed 0.3407: diff -0.0238 CI [-0.0341,-0.0139]
rho(P_exact, BM25 advantage) = -0.121 (p=0.007)


In [4]:
from rank_bm25 import BM25Okapi
clf_df = pd.concat([pd.read_csv(OUT / f"classifier_{n}.csv") for n in ["train", "val"]]).reset_index(drop=True)
bm = BM25Okapi([tokenize(t) for t in corpus["doc_text"]])
qc = st.encode([QP + t for t in clf_df["query"]], batch_size=64, normalize_embeddings=True, convert_to_numpy=True).astype("float32")

Gc = np.zeros((len(clf_df), len(alphas)))
for i, (qid, t) in enumerate(zip(clf_df["query_id"], clf_df["query"])):
    b = bm.get_scores(tokenize(t)); d = qc[i] @ doc_emb.T
    cand = np.union1d(np.argpartition(-b, K)[:K], np.argpartition(-d, K)[:K]); bn, dn = minmax(b[cand]), minmax(d[cand])
    for j, a in enumerate(alphas):
        s = a * bn + (1 - a) * dn
        Gc[i, j] = evaluate({qid: list(pids[cand[np.argsort(-s)[:10]]])}, qrels).iloc[0, 0]

tab = pd.DataFrame(Gc, columns=alphas).assign(intent=clf_df["intent"].values).groupby("intent").mean().round(3)
tab["n"] = clf_df["intent"].value_counts(); print(tab.to_string())
print("best alpha per class:", tab.drop(columns="n").idxmax(axis=1).to_dict())
isE = (clf_df["intent"] == "Exact").values.astype(float); adv = Gc[:, 9] - Gc[:, 1]
rho, p = spearmanr(isE, adv); print(f"\nrho(human Exact, BM25 advantage) = {rho:.3f} (p={p:.3f})")
fx = Gc[:, int(np.argmax(Gc.mean(0)))]
byc = np.zeros(len(clf_df))
for c in clf_df["intent"].unique():
    mk = (clf_df["intent"] == c).values; byc[mk] = Gc[mk][:, Gc[mk].mean(0).argmax()]
m, ci = boot(byc, fx); print(f"per-class best alpha minus best fixed (in-sample, perfect labels): {m:+.4f} CI [{ci[0]:+.4f},{ci[1]:+.4f}]")

            0.0    0.1    0.2    0.3    0.4    0.5    0.6    0.7    0.8    0.9    1.0    n
intent                                                                                    
Exact     0.501  0.515  0.539  0.547  0.535  0.516  0.500  0.486  0.471  0.459  0.440  112
Hybrid    0.319  0.327  0.343  0.337  0.335  0.324  0.303  0.289  0.276  0.264  0.252  184
Semantic  0.161  0.171  0.176  0.182  0.182  0.179  0.174  0.169  0.164  0.158  0.150  173
best alpha per class: {'Exact': 0.3, 'Hybrid': 0.2, 'Semantic': 0.3}

rho(human Exact, BM25 advantage) = -0.053 (p=0.250)
per-class best alpha minus best fixed (in-sample, perfect labels): +0.0021 CI [-0.0006,+0.0047]


In [6]:
test = eval_q[eval_q["role"] == "eval_test"]; trows = test.index.to_numpy(); tq = test["query_id"].tolist(); nt = len(tq)
VT = build_vecs(test, "eval_test"); pET = P.loc[trows, "pE_T"].values; fixa = np.full(nt, A_BGE)
COND = {"C1 BM25 only": (None, None), "C2 raw dense": ("raw", None), "C3 prose HyDE (dense only)": ("prose", None),
        "C4 spec HyDE + alpha=P(Exact)": ("spec", pET),
        "X1 raw + BM25": ("raw", fixa), "X2 prose+q dense only": ("prose_q", None), "X3 spec+q dense only": ("spec_q", None),
        "X4 prose+q + BM25": ("prose_q", fixa), "X5 spec+q + BM25": ("spec_q", fixa),
        "X6 spec+q + BM25 (a=P(Exact))": ("spec_q", pET), "X7 spec + BM25": ("spec", fixa), "X8 prose + BM25": ("prose", fixa)}
res = {k: run(trows, tq, VT, *v) for k, v in COND.items()}
sm = test["small"].values == 1
tab = pd.DataFrame({k: {"NDCG@10": r["ndcg@10"].mean(), "MRR@10": r["mrr@10"].mean(),
                        "NDCG small": r["ndcg@10"].values[sm].mean(), "MRR small": r["mrr@10"].values[sm].mean()} for k, r in res.items()}).T.round(4)
print(f"TEST bge (alpha={A_BGE}), n={nt}\n"); print(tab.to_string())

nd_ = lambda k: res[k]["ndcg@10"].values
print("\nPRIMARY contrasts, NDCG@10:")
for name, a, b in [("HyDE gain: X4 - X1", "X4 prose+q + BM25", "X1 raw + BM25"),
                   ("Novelty 1: X5 spec - X4 prose", "X5 spec+q + BM25", "X4 prose+q + BM25"),
                   ("Novelty 2: X6 a=P(Exact) - X5 fixed", "X6 spec+q + BM25 (a=P(Exact))", "X5 spec+q + BM25")]:
    m, ci = boot(nd_(a), nd_(b)); print(f"{name:40s} {m:+.4f} [{ci[0]:+.4f},{ci[1]:+.4f}]")
print("\nSECONDARY (descriptive):")
for name, a, b in [("X7 spec - X8 prose (fused, no query avg)", "X7 spec + BM25", "X8 prose + BM25"),
                   ("spec - prose, dense only (C4 vs C3 inputs)", "X3 spec+q dense only", "X2 prose+q dense only"),
                   ("X5 best fused - C1 BM25", "X5 spec+q + BM25", "C1 BM25 only")]:
    m, ci = boot(nd_(a), nd_(b)); print(f"{name:45s} {m:+.4f} [{ci[0]:+.4f},{ci[1]:+.4f}]")

old = pd.read_parquet(OUT / "final_test_per_query_v1.parquet").xs("ndcg@10", axis=1, level=1).reindex(tq)
print("\nbge vs MiniLM, same test queries (NDCG@10; alpha differs if A_BGE != 0.4):")
for nb, nm in [("X1 raw + BM25", "X1 raw dense + BM25 (a=0.4)"), ("X4 prose+q + BM25", "X4 prose+query HyDE + BM25 (a=0.4)"),
               ("X5 spec+q + BM25", "X5 spec+query HyDE + BM25 (a=0.4)"), ("C2 raw dense", "C2 raw dense")]:
    m, ci = boot(nd_(nb), old[nm].values); print(f"{nb:25s} bge {nd_(nb).mean():.4f} | MiniLM {old[nm].mean():.4f} | diff {m:+.4f} [{ci[0]:+.4f},{ci[1]:+.4f}]")
tab.to_csv(OUT / "final_test_results_bge_v1.csv")
pd.concat({k: r for k, r in res.items()}, axis=1).to_parquet(OUT / "final_test_per_query_bge_v1.parquet")

TEST bge (alpha=0.3), n=1295

                               NDCG@10  MRR@10  NDCG small  MRR small
C1 BM25 only                    0.2546  0.3780      0.2008     0.3103
C2 raw dense                    0.3045  0.4511      0.2559     0.3930
C3 prose HyDE (dense only)      0.2603  0.4050      0.2178     0.3416
C4 spec HyDE + alpha=P(Exact)   0.2714  0.4094      0.2198     0.3424
X1 raw + BM25                   0.3255  0.4756      0.2690     0.4041
X2 prose+q dense only           0.3115  0.4566      0.2621     0.3956
X3 spec+q dense only            0.3185  0.4676      0.2732     0.4210
X4 prose+q + BM25               0.3386  0.4896      0.2821     0.4230
X5 spec+q + BM25                0.3486  0.5053      0.2973     0.4515
X6 spec+q + BM25 (a=P(Exact))   0.3172  0.4636      0.2603     0.4005
X7 spec + BM25                  0.3187  0.4712      0.2674     0.4111
X8 prose + BM25                 0.3199  0.4715      0.2709     0.4066

PRIMARY contrasts, NDCG@10:
HyDE gain: X4 - X1             